In [ ]:
import os
from functools import reduce
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import col, lower, trim, regexp_replace, to_date, concat, lit, substring


postgres_jar = "/home/espinozaje/jars/postgresql-42.7.3.jar"
spark = SparkSession.builder.appName("Ingesta_Segura").config("spark.driver.memory", "4g").config("spark.jars", postgres_jar).getOrCreate()


schema = StructType([
StructField("DNI", StringType(), True), StructField("Asegurado", StringType(), True),
StructField("FecNac", StringType(), True), StructField("SEXO", StringType(), True),
StructField("CPP", StringType(), True), StructField("síntomas", StringType(), True),
StructField("enfermedad detectada", StringType(), True), StructField("Distrito", StringType(), True),
])
distritos = ["Callayuc", "Choros", "Cujillo", "Cutervo", "LaRamada", "Pimpingos", "Querocotillo", "SanAndresdeCutervo", "SanLuisdeLucma", "SanJuandeCutervo", "SantaCruz", "SantoDomingodelaCapilla", "SantoTomas", "Socota", "ToribioCasanova"]
base_path = "hdfs://localhost:9000/datalake/raw/clinic/historias_csv"
fecha = "ingestion_date=2025-09-29"

dfs = []
print("Leyendo...")
for distrito in distritos:
try:
df = spark.read.csv(f"{base_path}/{fecha}/distrito={distrito}/*.csv", header=True, schema=schema)
dfs.append(df)
except: pass

if dfs:
df_raw = reduce(DataFrame.unionByName, dfs)
print(f"Leídos {df_raw.count()} registros.")


df_clean = df_raw.withColumn("FecNac", to_date(col("FecNac"), "yyyy-MM-dd")).fillna("DESCONOCIDO").dropDuplicates(["DNI"]).drop("DNI")


IP_VM1 = "100.68.144.113"
db_url = f"jdbc:postgresql://{IP_VM1}:5432/postgres"

print(f"Enviando a BD Puente ({IP_VM1})...")

try:
df_clean.write \
.format("jdbc") \
.option("url", db_url) \
.option("dbtable", "pacientes_buffer") \
.option("user", "postgres") \
.option("password", "admin") \
.option("driver", "org.postgresql.Driver") \
.mode("overwrite") \
.save()
print("¡ÉXITO TOTAL! Datos en la base de datos.")
except Exception as e:
print(f"Error JDBC: {e}")
else:
print("No hay datos.")